# PortfolioAgent - Week 17 API Notebook
**Applied GenAI & Agentic AI Engineering Course - Week 17 - AgentForge**

An A2A-compatible agent that answers questions about the AgentForge capstone. The model calls tools over the capstone's `AGENTS.md`, week detail files, architecture and eval results, and every fact in an answer cites the source a tool returned. This notebook walks every API endpoint - the build cards, the agent's trace, a follow-up, the approval gate, A2A and the guards - mostly two ways: **cURL** and **Python (`requests`)**. The browser UI (`/`) and the rendered README (`/readme`) are pages, so open those in a browser.

---
### Before you start
1. Server running: `uvicorn app.main:app --reload --reload-dir app --port 8000`
2. `.env` filled in with `OPENAI_API_KEY` (see `.env.example`)
3. Run the **Setup** cell below once. The cells that ask the agent call the real model - each question is two or three model calls.

> **cURL on any OS:** the curl cells call `curl()`, a two-line helper from the Setup cell that runs the real `curl` binary (built into Windows 10+, macOS and Linux) with its arguments as a list - no shell quoting to get wrong, and every URL uses `BASE`.
>
> **Against your Render URL:** set `BASE` to it and set the `ADMIN_TOKEN` environment variable first - the `[admin]` routes (`/actions`, `/approve`, `/audit`, `/memory`) need it there. Locally, with no token set, every route is open.

In [1]:
# Setup - run this cell first
import os, requests, json, subprocess

BASE = 'http://localhost:8000'
TOKEN = os.environ.get('ADMIN_TOKEN')                       # only needed against a deployed URL
ADMIN = {'Authorization': f'Bearer {TOKEN}'} if TOKEN else {}
def curl(*args):
    """Run the real curl with these arguments - a list, so no shell quoting on any OS."""
    print(subprocess.run(['curl', '-s', *args], capture_output=True, text=True, encoding='utf-8').stdout)

print('Setup complete. BASE:', BASE, '| admin token:', 'set' if TOKEN else 'not set (local)')

Setup complete. BASE: http://localhost:8000 | admin token: not set (local)


---
## 1 - Health - `GET /health`
Both model tiers, both caps, and the price table.

In [2]:
curl(f'{BASE}/health')

{"status":"ok","small_model":"gpt-5.4-nano-2026-03-17","frontier_model":"gpt-5.4-mini-2026-03-17","max_iterations":8,"cost_ceiling_usd":0.05,"admin_token_required":false,"card_signed":true,"card_key":"ephemeral","pricing_per_1m":{"small":{"input":0.2,"output":1.25},"frontier":{"input":0.75,"output":4.5}}}


In [3]:
r = requests.get(f'{BASE}/health')
print('Status:', r.status_code)
print(json.dumps(r.json(), indent=2))

Status: 200
{
  "status": "ok",
  "small_model": "gpt-5.4-nano-2026-03-17",
  "frontier_model": "gpt-5.4-mini-2026-03-17",
  "max_iterations": 8,
  "cost_ceiling_usd": 0.05,
  "admin_token_required": false,
  "card_signed": true,
  "card_key": "ephemeral",
  "pricing_per_1m": {
    "small": {
      "input": 0.2,
      "output": 1.25
    },
    "frontier": {
      "input": 0.75,
      "output": 4.5
    }
  }
}


---
## 2 - Build cards - `GET /portfolio`
What the browser UI opens on: one card per week, parsed from the **Build history** in `data/AGENTS.md`, plus both eval gates' pass rates. Each card opens a read page, `GET /portfolio/{week}`. Text parsing only: no model call, no cost.

In [4]:
pf = requests.get(f'{BASE}/portfolio').json()
print(pf['title'], '-', len(pf['builds']), 'builds | gates:', pf['gates'])
for b in pf['builds']:
    print(f"W{b['week']:<3}{b['name']:28s}{BASE}/portfolio/{b['week']}")

AgentForge™ - 17 builds | gates: {'routing': {'score': 1.0, 'cases': 27, 'passed': 27}, 'tool_choice': {'score': 1.0, 'cases': 14, 'passed': 14}}
W1  ReleaseBot                  http://localhost:8000/portfolio/1
W2  IntentIQ                    http://localhost:8000/portfolio/2
W3  TicketStream                http://localhost:8000/portfolio/3
W4  KnowledgeVault              http://localhost:8000/portfolio/4
W5  CitationRAG                 http://localhost:8000/portfolio/5
W6  RAGOptimizer                http://localhost:8000/portfolio/6
W7  BreakRAG™                   http://localhost:8000/portfolio/7
W8  SpecialistTuner             http://localhost:8000/portfolio/8
W9  OpsAssist                   http://localhost:8000/portfolio/9
W10 TriageFlow™                 http://localhost:8000/portfolio/10
W11 AgentMesh™ + CohortMCP      http://localhost:8000/portfolio/11
W12 GuardianAI™                 http://localhost:8000/portfolio/12
W13 WorkbenchAI™                http://localhost:8000/portf

---
## 3 - A2A Agent Card - `GET /.well-known/agent-card.json`
The discovery document any A2A client reads first. It advertises two skills (`capstone_qa`, `request_intro`) and, under `supportedInterfaces`, **where and how to call the agent**: `/a2a`, JSON-RPC, protocol 1.0.

In [5]:
curl(f'{BASE}/.well-known/agent-card.json')

{"name": "agentforge-portfolio", "description": "Answers questions about a student's capstone - what each week built, its results and its design decisions - by calling tools over the project's AGENTS.md, architecture and eval results, and cites the source of every fact. Routes cheap questions to a small model and escalates reasoning to a frontier model; an intro request pauses for the student's approval.", "supportedInterfaces": [{"url": "http://localhost:8000/a2a", "protocolBinding": "JSONRPC", "protocolVersion": "1.0"}], "version": "1.0.0", "documentationUrl": "http://localhost:8000/readme", "capabilities": {"streaming": false, "pushNotifications": false, "extendedAgentCard": false}, "defaultInputModes": ["text/plain"], "defaultOutputModes": ["text/plain"], "skills": [{"id": "capstone_qa", "name": "Capstone Q&A", "description": "Answer a question about what the capstone built, week by week, its eval results or its design - looked up with tools, every fact cited.", "tags": ["capstone"

In [6]:
card = requests.get(f'{BASE}/.well-known/agent-card.json').json()
print('Agent:', card['name'])
print('Skills:', [s['id'] for s in card['skills']])
print(json.dumps(card, indent=2))

Agent: agentforge-portfolio
Skills: ['capstone_qa', 'request_intro']
{
  "name": "agentforge-portfolio",
  "description": "Answers questions about a student's capstone - what each week built, its results and its design decisions - by calling tools over the project's AGENTS.md, architecture and eval results, and cites the source of every fact. Routes cheap questions to a small model and escalates reasoning to a frontier model; an intro request pauses for the student's approval.",
  "supportedInterfaces": [
    {
      "url": "http://localhost:8000/a2a",
      "protocolBinding": "JSONRPC",
      "protocolVersion": "1.0"
    }
  ],
  "version": "1.0.0",
  "documentationUrl": "http://localhost:8000/readme",
  "capabilities": {
    "streaming": false,
    "pushNotifications": false,
    "extendedAgentCard": false
  },
  "defaultInputModes": [
    "text/plain"
  ],
  "defaultOutputModes": [
    "text/plain"
  ],
  "skills": [
    {
      "id": "capstone_qa",
      "name": "Capstone Q&A",
   

---
## 4 - A2A - talk to the agent the way another agent would
JSON-RPC `SendMessage` to the URL the card advertises. The reply is a `Message` from `ROLE_AGENT`: one text part, plus `metadata` with the tier, model, cost, the tools the agent called and the citations it verified - the same loop as `POST /ask`.

In [7]:
curl('-X', 'POST', f'{BASE}/a2a', '-H', 'Content-Type: application/json', '-H', 'A2A-Version: 1.0',
     '-d', json.dumps({'jsonrpc': '2.0', 'id': 1, 'method': 'SendMessage', 'params': {'message': {
         'messageId': 'm-1', 'role': 'ROLE_USER', 'parts': [{'text': 'What is AgentForge?'}]}}}))

{"jsonrpc":"2.0","id":1,"result":{"message":{"messageId":"cc0c5ce2b64f46c682d8ca41152e12ec","contextId":"1e92a6251628497393f9ef1058c4aae7","role":"ROLE_AGENT","parts":[{"text":"AgentForge™ is an engineering-operations copilot for a platform team. It answers questions grounded in runbooks and design documents with checked citations, triages incoming incidents with a small team of agents, proposes remediations that wait at a human approval gate, routes every model call to the cheapest capable tier, and is deployed behind CI with an eval gate, tracing, reliability guards, and a cost report. The PortfolioAgent is its public front door. [AGENTS.md · overview]"}],"metadata":{"tier":"small","model":"gpt-5.4-nano-2026-03-17","grounded":true,"skillMatched":"capstone_qa","costUsd":0.000617,"savedUsd":0.001677,"toolsCalled":[{"tool":"get_architecture","args":{"section":"overview"},"success":true}],"citations":["AGENTS.md · overview"],"pendingActionId":null}}}}


In [8]:
card = requests.get(f'{BASE}/.well-known/agent-card.json').json()
a2a_url = card['supportedInterfaces'][0]['url']             # discovered, not hard-coded
url = a2a_url.replace('http://localhost:8000', BASE)        # same path on any BASE
reply = requests.post(url, headers={'A2A-Version': '1.0'}, json={
    'jsonrpc': '2.0', 'id': 1, 'method': 'SendMessage',
    'params': {'message': {'messageId': 'm-1', 'role': 'ROLE_USER',
                           'parts': [{'text': 'What is AgentForge?'}]}}}).json()
msg = reply['result']['message']
print('role:', msg['role'], '| tier:', msg['metadata']['tier'])
print('tools:', msg['metadata']['toolsCalled'])
print('cited:', msg['metadata']['citations'])
print(msg['parts'][0]['text'])

role: ROLE_AGENT | tier: small
tools: [{'tool': 'get_architecture', 'args': {'section': 'overview'}, 'success': True}]
cited: ['AGENTS.md · overview']
AgentForge is “an engineering-operations copilot for a platform team” that answers questions grounded in runbooks/design docs with checked citations, triages incidents with a small team of agents, proposes remediations that wait at a human approval gate, and routes each model call to the cheapest capable tier. It’s deployed behind CI with an eval gate, tracing, reliability guards, and a cost report, with the PortfolioAgent as its public front door. [AGENTS.md · overview]


---
## 5 - A2A - the other core methods answer, with the spec's errors
This agent replies with a `Message` and never creates a task, so the task methods answer the way A2A 1.0 says an agent like this must: an empty `ListTasks` page, `TaskNotFound` (-32001), `UnsupportedOperation` for streaming (-32004). A client asking for another protocol version gets `VersionNotSupported` (-32009) - and so does one that sends no `A2A-Version` header, because the spec reads a missing header as 0.3.

In [9]:
def rpc(method, params=None, version='1.0'):
    body = {'jsonrpc': '2.0', 'id': 7, 'method': method, 'params': params or {}}
    return requests.post(f'{BASE}/a2a', json=body, headers={'A2A-Version': version}).json()

print('ListTasks           :', rpc('ListTasks', {'pageSize': 10})['result'])
for method in ('GetTask', 'SendStreamingMessage', 'GetExtendedAgentCard'):
    err = rpc(method, {'id': 't-1'})['error']
    print(f'{method:20s}:', err['code'], err['message'])
err = rpc('SendMessage', {'message': {'messageId': 'm', 'role': 'ROLE_USER',
                                      'parts': [{'text': 'What is the cost ceiling?'}]}}, version='0.3')['error']
print('A2A-Version 0.3     :', err['code'], err['message'])
err = requests.post(f'{BASE}/a2a', json={'jsonrpc': '2.0', 'id': 8, 'method': 'ListTasks'}).json()['error']
print('no A2A-Version      :', err['code'], err['message'])

ListTasks           : {'tasks': [], 'totalSize': 0, 'pageSize': 10, 'nextPageToken': ''}


GetTask             : -32001 Task not found


SendStreamingMessage: -32004 Streaming is not supported


GetExtendedAgentCard: -32007 No extended Agent Card


A2A-Version 0.3     : -32009 A2A 0.3 is not supported


no A2A-Version      : -32009 A2A 0.3 is not supported


---
## 6 - Ask a lookup - the SMALL tier, and the tools the model chose
`What did you build in Week 4?` has no reasoning cue, so the classifier routes it to the small tier (gpt-5.4-nano). The model then **chooses** its tools by function calling - here `get_build(4)` - and must cite what they returned. `grounded` is true only if every citation matches a source a tool returned in this run.

In [10]:
curl('-X', 'POST', f'{BASE}/ask', '-H', 'Content-Type: application/json',
     '-d', json.dumps({'question': 'What did you build in Week 4?'}))

{"answer":"In Week 4, the student built **KnowledgeVault**: a multimodal ingestion pipeline for engineering PDFs where a vision model describes figures and tables are converted into row-like records; prose is chunked into **300-token children** inside **1,500-token parents** (only children are embedded). Embeddings use **`text-embedding-3-large` (3,072 dims)** stored in **Qdrant**, and retrieval combines **dense + BM25** fused with **RRF**. [AGENTS.md · W4]","model":"gpt-5.4-nano-2026-03-17","tier":"small","complexity":"simple","grounded":true,"skill_matched":"capstone_qa","routed_up":false,"tools_called":[{"tool":"get_build","args":{"week":4},"success":true,"source":"AGENTS.md · W4","error":null,"content":"W4 · KnowledgeVault\nmultimodal ingestion of engineering PDFs: a vision model forced through function-calling describes figures, tables become rows, prose is chunked into 300-token children inside 1,500-token parents (only children are embedded). Embeddings are `text-embedding-3-lar

In [11]:
d = requests.post(f'{BASE}/ask', json={'question': 'What did you build in Week 4?'}).json()
print('tier :', d['tier'], '| model:', d['model'], '| model calls:', d['model_calls'])
print('trace:', ' -> '.join(f"{s['tool']}({', '.join(map(str, s['args'].values()))})" for s in d['tools_called']))
print('cited:', d['citations'], '| invented:', d['unverified_citations'], '| grounded:', d['grounded'])
print('cost : $%.6f  (saved $%.6f vs the frontier)' % (d['cost_usd'], d['saved_usd']))
print(d['answer'])

tier : small | model: gpt-5.4-nano-2026-03-17 | model calls: 2
trace: get_build(4)
cited: ['AGENTS.md · W4'] | invented: [] | grounded: True
cost : $0.000645  (saved $0.001748 vs the frontier)
In **Week 4**, the student built **KnowledgeVault**: a system for **multimodal ingestion of engineering PDFs**. It uses a **vision model** to describe figures via **function-calling**, converts tables into **row records**, and **chunks prose** into **300-token children inside 1,500-token parents** (embedding only the children). For retrieval, it stores embeddings in **Qdrant** using `text-embedding-3-large` (**3,072 dims**) and performs **dense + BM25 fusion with RRF**. [AGENTS.md · W4]


---
## 7 - What the agent retrieved
Each step of the trace carries the text that tool call pulled - the context the model answered from. The UI shows the same thing under **Retrieved context**.

In [12]:
for s in d['tools_called']:
    used = 'cited' if s['source'] in d['citations'] else 'not cited'
    print(f"--- {s['tool']}({s['args']}) -> {s['source']} [{used}]")
    print((s['content'] or s['error'] or '')[:400])

--- get_build({'week': 4}) -> AGENTS.md · W4 [cited]
W4 · KnowledgeVault
multimodal ingestion of engineering PDFs: a vision model forced through function-calling describes figures, tables become rows, prose is chunked into 300-token children inside 1,500-token parents (only children are embedded). Embeddings are `text-embedding-3-large` (3,072 dims) in Qdrant; retrieval is dense + BM25 fused with RRF.


---
## 8 - A reasoning question - the FRONTIER tier, several tools
A `why` / `compare` question escalates to gpt-5.4-mini. Comparing two weeks, the model typically calls a tool per week - and may go deeper with `get_week_details` or `get_architecture(week=...)`.

In [13]:
d = requests.post(f'{BASE}/ask', json={'question': 'Compare what Week 5 and Week 6 built.'}).json()
print('tier :', d['tier'], '| model:', d['model'], '| model calls:', d['model_calls'])
print('trace:', [f"{s['tool']}({s['args']})" for s in d['tools_called']])
print('cited:', d['citations'])
print(d['answer'])

tier : frontier | model: gpt-5.4-mini-2026-03-17 | model calls: 2
trace: ["get_build({'week': 5})", "get_build({'week': 6})"]
cited: ['AGENTS.md · W5', 'AGENTS.md · W6']
Week 5 built **CitationRAG**, which focused on grounded answers with citations: retrieval was gated before the model ran, every citation id and quote was checked deterministically, and it used a fixed refusal string when gates failed [AGENTS.md · W5].

Week 6 built **RAGOptimizer**, which added retrieval and ranking improvements: a HyDE nano probe whose candidates were unioned with the original query’s candidates, a CPU cross-encoder reranker scoring all 30 pairs in one batch, and extractive compression that kept 80% while preserving citation markers [AGENTS.md · W6].

So, Week 5 was about **safe, cited answering**, while Week 6 was about **improving retrieval quality and efficiency** on top of that [AGENTS.md · W5] [AGENTS.md · W6].


---
## 9 - A follow-up - the last turns travel with the question
The browser keeps the last 3 turns and sends them as `history`; the server stores none of it. The tier is decided on the NEW question only.

In [14]:
first = requests.post(f'{BASE}/ask', json={'question': 'What did you build in Week 16?'}).json()
history = [{'question': 'What did you build in Week 16?', 'answer': first['answer']}]
d = requests.post(f'{BASE}/ask', json={'question': 'Why did you choose that design?',
                                      'history': history}).json()
print('trace:', [f"{s['tool']}({s['args']})" for s in d['tools_called']])
print(d['answer'])

trace: ["get_week_details({'week': 16})"]
The student chose this design to **minimize cost without sacrificing accuracy**: the router sends each task to the cheapest tier that should handle it, and it **routes up when confidence is below 0.60** because a wrong cheap answer is more expensive than a right expensive one. [weeks/w16.md]  

They also chose to **bill by the served model id** so silent model re-pins would show up as lost savings, and to **count tokens with the tokenizer** because character-based estimates can undercount dense text like logs. [weeks/w16.md]  

For the SRE bot, they used **SQLite for paused actions** so approvals survive restarts without needing a server. [weeks/w16.md]


---
## 10 - The approval gate - an intro request waits for the student
Ask to get in touch and the model calls `request_intro`. It creates an action at `input-required` and does **nothing else**; only the student (the admin token on a deploy) decides it - from the inbox, `GET /actions`, then `POST /approve`. Approval is state the system owns: no sentence from the visitor or the model can approve it.

In [15]:
d = requests.post(f'{BASE}/ask', json={'question': "I'm hiring an AI engineer at Acme - please "
                                      "ask the student to contact me at jane@example.com."}).json()
pending = d['pending_action']
print('agent  :', d['answer'])
assert pending, 'the model did not call request_intro this time - run the cell again'
print('status :', pending['status'], '| id:', pending['id'])
print('inbox  :', [a['id'] for a in requests.get(f'{BASE}/actions', headers=ADMIN).json()])
done = requests.post(f'{BASE}/approve', json={'action_id': pending['id'], 'approve': True}, headers=ADMIN).json()
print('after approve:', done['status'], '-', done['result'])

agent  : The request is waiting for the student’s approval.
status : input-required | id: 25126121


inbox  : ['25126121']


after approve: executed - Intro accepted by owner: the student will reply to Jane.


---
## 11 - Failure - too-short question, or too much history (422)
`min_length=8`, and at most 3 history turns of 2,000 characters each; both rejected before any model call.

In [16]:
print('hi        :', requests.post(f'{BASE}/ask', json={'question': 'hi'}).status_code, '(expected 422)')
turn = {'question': 'q', 'answer': 'a'}
r = requests.post(f'{BASE}/ask', json={'question': 'What did you build in Week 4?', 'history': [turn] * 4})
print('4 turns   :', r.status_code, '(expected 422)')

hi        : 422 (expected 422)


4 turns   : 422 (expected 422)


---
## 12 - Failure - off-topic question (no tool, out of scope)
Not about the capstone: the model calls no tool and replies with the `OUT_OF_SCOPE` sentinel, so `grounded` is false.

In [17]:
d = requests.post(f'{BASE}/ask', json={'question': 'What is the capital of France?'}).json()
print('tools   :', d['tools_called'], '| grounded:', d['grounded'])
print('answer  :', d['answer'])

tools   : [] | grounded: False
answer  : OUT_OF_SCOPE — I only answer questions about the student’s capstone portfolio (AgentForge™).


---
## 13 - Failure - oversized input (413, the cost ceiling)
A deliberately enormous input. `app/budget.py` prices the request before the model call and refuses it. **Nothing is spent.**

In [18]:
huge = 'Describe your architecture. ' * 45000   # 28 x 45,000 = 1,260,000 characters
r = requests.post(f'{BASE}/ask', json={'question': huge})
print('Status:', r.status_code, '(expected 413)')
print(json.dumps(r.json(), indent=2))

Status: 413 (expected 413)
{
  "detail": {
    "error": "cost_ceiling_exceeded",
    "projected_usd": 0.141747,
    "ceiling_usd": 0.05
  }
}


---
## 14 - Memory - the audit log is scoped per user
Ask as `alice`, then read her audit log - each line records the tools the agent called. Another user sees nothing of hers.

In [19]:
requests.post(f'{BASE}/ask', json={'question':'What is the cost ceiling?','user_id':'alice'})
print('alice:', requests.get(f'{BASE}/audit/alice', headers=ADMIN).json())
print('bob  :', requests.get(f'{BASE}/audit/bob', headers=ADMIN).json())

alice: [{'user_id': 'alice', 'session_id': 'default', 'kind': 'ask', 'detail': '[small] What is the cost ceiling? | get_architecture(rules)'}]


bob  : []


---
## 15 - Memory - remember, recall, and the three boundary probes
Memory is keyed by user, then session. A different user, or the same user in another session, gets `found: false` - the same answer as a key that never existed, so nothing leaks. A delete is a hard delete.

In [20]:
pref = {'user_id': 'alice', 'session_id': 's1', 'key': 'pref'}
print('write          :', requests.post(f'{BASE}/memory', json={**pref, 'value': 'dark mode'}, headers=ADMIN).json())
print('alice / s1     :', requests.get(f'{BASE}/memory', params=pref, headers=ADMIN).json()['value'])
print('bob   / s1     :', requests.get(f'{BASE}/memory', params={**pref, 'user_id': 'bob'}, headers=ADMIN).json()['found'])
print('alice / s2     :', requests.get(f'{BASE}/memory', params={**pref, 'session_id': 's2'}, headers=ADMIN).json()['found'])
print('delete         :', requests.delete(f'{BASE}/memory', json=pref, headers=ADMIN).json())
print('after delete   :', requests.get(f'{BASE}/memory', params=pref, headers=ADMIN).json()['found'])

write          : {'stored': True, 'user_id': 'alice', 'session_id': 's1', 'key': 'pref'}


alice / s1     : dark mode


bob   / s1     : False


alice / s2     : False


delete         : {'deleted': True, 'key': 'pref'}


after delete   : False


---
## 16 - Public-deploy guards - what a shared URL needs
A link on your resume means strangers spend **your** key. Four guards in `app/guard.py`: the admin token on `[admin]` routes (required on Render - the app will not boot there without it; the owner pastes it on the `/admin` page), a per-client limit on POSTs (`RATE_LIMIT_PER_MINUTE`, default 30), a daily model-spend budget across every caller and every call of every loop (`DAILY_BUDGET_USD`, default $1.00), and a 2 MiB body limit (413 before the body is read). The cell below trips the rate limit on purpose with a free POST (an A2A `ListTasks`) - run it last, or wait a minute before the next POST.

In [21]:
cheap = {'jsonrpc': '2.0', 'id': 1, 'method': 'ListTasks'}
# One Session = one kept-alive connection. Without it, each POST opens a new connection - and on
# Windows `localhost` tries IPv6 first (~2 s each), too slow to fit 30 POSTs in a minute.
with requests.Session() as s:
    for n in range(1, 61):
        r = s.post(f'{BASE}/a2a', json=cheap, headers={'A2A-Version': '1.0'})
        if r.status_code == 429:
            print(f'POST #{n} in this burst -> 429', r.json()['detail'], '| Retry-After:', r.headers['Retry-After'])
            break
    else:
        print('no 429 - is RATE_LIMIT_PER_MINUTE set to 0?')

POST #31 in this burst -> 429 {'error': 'rate_limited', 'limit_per_minute': 30, 'retry_after_s': 60} | Retry-After: 60


---
## 17 - OpenAPI / Swagger Docs
FastAPI auto-generates interactive docs.

In [22]:
from IPython.display import display, HTML
display(HTML('<a href="http://localhost:8000/docs" target="_blank">Open Swagger UI: http://localhost:8000/docs</a>'))